# Traditional models on the shared prepared split

Adapted from the supplied `baseline.ipynb`. The data directory now points to
`outputs_hier_strategy4`, the same split as `model.ipynb`. Original outputs were
removed. The adapted code body was executed locally and aggregate metrics are in `results/shared_split_traditional_*`. Historical scores came from
an earlier 605-post test set. Re-run before making a matched-split comparison.
Run `python scripts/prepare_experiment.py Dataset` first.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from the repository root; this also supports the default
# notebook-directory working directory without personal absolute paths.
if Path.cwd().name == "notebooks" and (Path.cwd().parent / "README.md").is_file():
    os.chdir(Path.cwd().parent)
print("Working directory: repository root" if (Path.cwd()/"README.md").is_file() else "Check working directory")


In [ ]:
# Cell 1: 导入必要的库
import os
import json
import numpy as np
import pandas as pd
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, hamming_loss, f1_score
from sklearn.pipeline import Pipeline

# 忽略不必要的警告
warnings.filterwarnings('ignore')

print("Libraries loaded.")

# Cell 2: 加载数据 (直接读取 scripts/prepare_experiment.py 处理好的数据)
# ---------------------------------------------------------------------
# 确保此 Notebook 与 outputs_hier 文件夹在同一目录下
# ---------------------------------------------------------------------
DATA_DIR = "outputs_hier_strategy4" 
TRAIN_PATH = os.path.join(DATA_DIR, "train.jsonl")
TEST_PATH  = os.path.join(DATA_DIR, "test.jsonl")
LABELS_PATH = os.path.join(DATA_DIR, "labels.json")

# 检查文件是否存在
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"找不到 {TRAIN_PATH}。请先运行 scripts/prepare_experiment.py Dataset 生成数据。")

print("正在加载数据...")
train_df = pd.read_json(TRAIN_PATH, lines=True)
test_df  = pd.read_json(TEST_PATH, lines=True)

# 加载标签名称 (保持顺序一致)
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    LABELS = json.load(f)

print(f"训练集样本数: {len(train_df)}")
print(f"测试集样本数: {len(test_df)}")
print(f"标签列表 ({len(LABELS)}): {LABELS}")

# ---------------------------------------------------------------------
# 数据预处理：转换为 Scikit-Learn 需要的格式
# X: 文本列表
# Y: 二维 Numpy 数组 (样本数 x 标签数)
# ---------------------------------------------------------------------
X_train = train_df["text_clean"].fillna("").astype(str).values
X_test = test_df["text_clean"].fillna("").astype(str).values

# 将 list 形式的 y (e.g., [1, 0, 1...]) 堆叠成矩阵
Y_train = np.vstack(train_df["y"].values)
Y_test = np.vstack(test_df["y"].values)

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_test shape:  {X_test.shape},  Y_test shape:  {Y_test.shape}")

# Cell 3: 定义基线模型
# 根据 Slide 9 文献综述，常用基线包括 SVM, Naive Bayes, Logistic Regression
# 使用 TF-IDF 提取特征，OneVsRestClassifier 处理多标签分类

baseline_models = {
    "SVM (TF-IDF)": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1,2))),
        ('clf', OneVsRestClassifier(LinearSVC(class_weight='balanced', random_state=42, dual='auto')))
    ]),
    
    "Logistic Regression (TF-IDF)": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1,2))),
        ('clf', OneVsRestClassifier(LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)))
    ]),
    
    "Naive Bayes (TF-IDF)": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1,2))),
        ('clf', OneVsRestClassifier(MultinomialNB()))
    ])
}

# Cell 4: 训练、评估并汇总结果
# ---------------------------------------------------------------------
# 核心逻辑：遍历所有模型，将结果收集到一个统一的列表中
# ---------------------------------------------------------------------

all_metrics_data = [] # 用于存储最终的大表数据

print("开始训练基线模型...\n")

for model_name, pipeline in baseline_models.items():
    print(f"正在运行: {model_name} ...")
    
    # 1. 训练
    pipeline.fit(X_train, Y_train)
    
    # 2. 预测
    y_pred = pipeline.predict(X_test)
    
    # 3. 计算总体指标
    h_loss = hamming_loss(Y_test, y_pred)
    # exact_acc = accuracy_score(Y_test, y_pred)
    micro_f1 = f1_score(Y_test, y_pred, average='micro')
    
    # 4. 生成详细的分标签报告 (dict 格式)
    report_dict = classification_report(
        Y_test, y_pred, 
        target_names=LABELS, 
        output_dict=True, 
        zero_division=0
    )
    
    # 5. 将报告解析为行数据，并添加 'Model' 列
    # 遍历每个标签 (Anger, Sadness, etc.)
    for label in LABELS:
        metrics = report_dict[label]
        all_metrics_data.append({
            "Model": model_name,
            "Category": "Per-Label",  # 标记为具体标签
            "Label/Metric Name": label,
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1-Score": metrics["f1-score"],
            "Support": metrics["support"]
        })
    
    # 遍历总体平均指标 (micro avg, macro avg, etc.)
    for avg_name in ['micro avg', 'macro avg', 'weighted avg', 'samples avg']:
        if avg_name in report_dict:
            metrics = report_dict[avg_name]
            all_metrics_data.append({
                "Model": model_name,
                "Category": "Average", # 标记为平均值
                "Label/Metric Name": avg_name,
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "F1-Score": metrics["f1-score"],
                "Support": metrics["support"]
            })
            
    # 添加 Hamming Loss 等额外指标 (作为单独的行)
    all_metrics_data.append({
        "Model": model_name,
        "Category": "Global",
        "Label/Metric Name": "Hamming Loss",
        "Precision": None, "Recall": None, "F1-Score": h_loss, "Support": None
    })
    
    print(f"  -> 完成。Micro F1: {micro_f1:.4f}")

print("\n所有模型评估完毕。")

# Cell 5: 结果导出与展示
# ---------------------------------------------------------------------
# 将收集到的数据转换为 DataFrame 并保存为 CSV
# ---------------------------------------------------------------------

# 创建 DataFrame
df_results = pd.DataFrame(all_metrics_data)

# 调整列顺序，使其更符合阅读习惯
cols_order = ["Model", "Category", "Label/Metric Name", "Precision", "Recall", "F1-Score", "Support"]
df_results = df_results[cols_order]

# 定义输出路径 (与 scripts/prepare_experiment.py 的输出放在一起)
output_file = os.path.join(DATA_DIR, "baselines_performance_report.csv")

# 保存
df_results.to_csv(output_file, index=False)

print("\n" + "="*60)
print(f"✅ 基线模型性能报告已保存至: {output_file}")
print("="*60)

# 预览：只显示 Macro F1 和 Micro F1 的对比
print("\n=== 总体性能概览 (Preview) ===")
summary_view = df_results[df_results["Label/Metric Name"].isin(["micro avg", "macro avg"])]
display(summary_view)

# 预览：查看某个具体标签 (例如 'depression' 或列表中的第一个) 的对比
first_label = LABELS[0]
print(f"\n=== 标签 '{first_label}' 性能对比 ===")
label_view = df_results[df_results["Label/Metric Name"] == first_label]
display(label_view)